# Experiment 4 — Variable importance and interpretation

## Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import negolin_utils as nu
nu.setup_matplotlib()

# Override paths if your data lives elsewhere:
# nu.RAND_BUYER_DIR = Path('/abs/path/to/0421_redo_previous_exp/results')

# Load data and apply v7 preprocessing
raw = nu.load_all_fixed_policies(nu.RAND_BUYER_DIR)
_, turns = nu.preprocess_turns(raw)
POLICIES = list(nu.POLICIES)

# Load Stage-2 linear model selection from Exp02 (if present)
try:
    selected = pd.read_csv('selected_linear_models.csv')
    SELECTED = {row['policy']: (row['feature'], row['model'], row.get('alpha', np.nan))
                for _, row in selected.iterrows()}
    print('Loaded per-policy (feature, model, alpha) from Exp02.')
except FileNotFoundError:
    # Fall back to a reasonable default (Feature II, Ridge) if Exp02 hasn't run
    SELECTED = {p: ('II', 'Ridge', 1.0) for p in POLICIES}
    print('selected_linear_models.csv not found; using default (Feature II, Ridge).')

**Goal.** For each policy's Stage-2 linear model, report:

- standardized coefficients (the effect of a 1-SD change in each feature
  on the logit of $\tilde S_t$)
- drop-column importance (CV $\Delta$MAE when each feature is removed)
- permutation importance (CV $\Delta$MAE when a feature column is shuffled)

A natural-language interpretation per policy is auto-generated from the
top-magnitude coefficients.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge

def fit_final_linear(policy):
    feat, mdl, alpha = SELECTED[policy]
    meta, X, y, cols = nu.build_design_matrix(turns, policy, feat)
    scaler = StandardScaler().fit(X)
    Xs = scaler.transform(X)
    y_logit = nu.logit_clip(y)
    if mdl == 'OLS':
        reg = LinearRegression().fit(Xs, y_logit)
    else:
        a = float(alpha) if np.isfinite(alpha) else 1.0
        reg = Ridge(alpha=a).fit(Xs, y_logit)
    return cols, reg.coef_, float(reg.intercept_), feat, mdl, meta, X, y

coef_rows = []
for p in POLICIES:
    cols, coef, intercept, feat, mdl, _, _, _ = fit_final_linear(p)
    for c, v in zip(cols, coef):
        coef_rows.append({'policy': p, 'feature_map': feat, 'term': c, 'coef': v})
    coef_rows.append({'policy': p, 'feature_map': feat, 'term': 'intercept', 'coef': intercept})

coef_tbl = (pd.DataFrame(coef_rows)
              .pivot_table(index='term', columns='policy', values='coef', aggfunc='first')
              .reindex(columns=POLICIES))
coef_tbl.round(3)

### Coefficient heatmap (standardized, logit scale)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 0.3*len(coef_tbl) + 1))
vmax = float(np.nanmax(np.abs(coef_tbl.to_numpy())))
im = ax.imshow(coef_tbl.fillna(0).to_numpy(), cmap='RdBu_r',
               vmin=-vmax, vmax=vmax, aspect='auto')
ax.set_xticks(range(len(coef_tbl.columns))); ax.set_xticklabels(coef_tbl.columns)
ax.set_yticks(range(len(coef_tbl)));         ax.set_yticklabels(coef_tbl.index)
plt.colorbar(im, ax=ax, fraction=0.04)
for i in range(len(coef_tbl)):
    for j in range(len(coef_tbl.columns)):
        v = coef_tbl.iat[i, j]
        if np.isfinite(v):
            ax.text(j, i, f'{v:+.2f}', ha='center', va='center', fontsize=8,
                    color='white' if abs(v) > vmax*0.6 else 'black')
ax.set_title('Standardized linear coefficients (logit scale)')
plt.tight_layout()
plt.savefig('figE4_coef_heatmap.png', dpi=140, bbox_inches='tight')
plt.show()

### Drop-column importance

In [ ]:
def drop_col_importance(policy, col_to_drop):
    feat, mdl, alpha = SELECTED[policy]
    meta, X, y, cols = nu.build_design_matrix(turns, policy, feat)
    j = cols.index(col_to_drop)
    X_drop = np.delete(X, j, axis=1)
    pred, _ = nu.grouped_cv_predict(meta, X_drop, y, mdl,
                                    n_splits=nu.OUTER_FOLDS,
                                    fixed_alpha=float(alpha) if (mdl == 'Ridge' and np.isfinite(alpha)) else None)
    return nu.cv_metrics_full(pred)['mae_norm']

imp_rows = []
for p in POLICIES:
    feat, mdl, alpha = SELECTED[p]
    meta, X, y, cols = nu.build_design_matrix(turns, p, feat)
    base_pred, _ = nu.grouped_cv_predict(meta, X, y, mdl, n_splits=nu.OUTER_FOLDS,
                                          fixed_alpha=float(alpha) if (mdl == 'Ridge' and np.isfinite(alpha)) else None)
    base_mae = nu.cv_metrics_full(base_pred)['mae_norm']
    for c in cols:
        delta = drop_col_importance(p, c) - base_mae
        imp_rows.append({'policy': p, 'term': c, 'delta_mae': delta})

imp_tbl = (pd.DataFrame(imp_rows)
            .pivot_table(index='term', columns='policy',
                          values='delta_mae', aggfunc='first')
            .reindex(columns=POLICIES))
imp_tbl.round(4)